In [2]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

In [4]:
# Cell 2: Load data and best params
X_train = joblib.load('../data/processed/X_train_scaled.pkl')
X_train_raw = joblib.load('../data/processed/x_train.pkl')
X_test = joblib.load('../data/processed/X_test_scaled.pkl')
X_test_raw = joblib.load('../data/processed/x_test.pkl')
y_train = joblib.load('../data/processed/y_train.pkl')
y_test = joblib.load('../data/processed/y_test.pkl')

best_params = joblib.load('../models/optuna_best_params.pkl')

print("Loaded optimized parameters")

Loaded optimized parameters


In [5]:
# Cell 4: Train XGBoost
print("Training XGBoost...")
xgb_params = best_params['xgb_best_params']
xgb_params['random_state'] = 42
xgb_params['eval_metric'] = 'logloss'
xgb_model = xgb.XGBClassifier(**xgb_params)
xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

xgb_acc = accuracy_score(y_test, xgb_pred)
xgb_f1 = f1_score(y_test, xgb_pred)
xgb_auc = roc_auc_score(y_test, xgb_proba)

print(f"XGBoost - Acc: {xgb_acc:.4f}, F1: {xgb_f1:.4f}, AUC: {xgb_auc:.4f}")

Training XGBoost...
XGBoost - Acc: 0.7602, F1: 0.6092, AUC: 0.8048


In [6]:
# Cell 5: Train LightGBM
print("Training LightGBM...")
lgb_params = best_params['lgb_best_params']
lgb_params['random_state'] = 42
lgb_params['verbose'] = -1
lgb_model = lgb.LGBMClassifier(**lgb_params)
lgb_model.fit(X_train, y_train)

lgb_pred = lgb_model.predict(X_test)
lgb_proba = lgb_model.predict_proba(X_test)[:, 1]

lgb_acc = accuracy_score(y_test, lgb_pred)
lgb_f1 = f1_score(y_test, lgb_pred)
lgb_auc = roc_auc_score(y_test, lgb_proba)

print(f"LightGBM - Acc: {lgb_acc:.4f}, F1: {lgb_f1:.4f}, AUC: {lgb_auc:.4f}")


Training LightGBM...
LightGBM - Acc: 0.7602, F1: 0.6099, AUC: 0.8045


In [15]:
# Cell 6: Train CatBoost
print("Training CatBoost...")
cat_params = best_params['cat_best_params']
cat_params['random_seed'] = 42
cat_params['verbose'] = False
cat_model = CatBoostClassifier(**cat_params)
cat_model.fit(X_train_raw, y_train)

cat_pred = cat_model.predict(X_test_raw)
cat_proba = cat_model.predict_proba(X_test_raw)[:, 1]

cat_acc = accuracy_score(y_test, cat_pred)
cat_f1 = f1_score(y_test, cat_pred)
cat_auc = roc_auc_score(y_test, cat_proba)

print(f"CatBoost - Acc: {cat_acc:.4f}, F1: {cat_f1:.4f}, AUC: {cat_auc:.4f}")


Training CatBoost...
CatBoost - Acc: 0.7601, F1: 0.6091, AUC: 0.8044


In [17]:
# Cell 7: Stacking with 4 base models
print("\nBuilding stacking ensemble...")

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

xgb_oof = np.zeros(len(X_train))
lgb_oof = np.zeros(len(X_train))
cat_oof = np.zeros(len(X_train))

xgb_test_preds = np.zeros(len(X_test))
lgb_test_preds = np.zeros(len(X_test))
cat_test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"Fold {fold + 1}/{n_splits}")
    
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr = y_train.iloc[train_idx]
    
    # XGBoost fold
    xgb_fold = xgb.XGBClassifier(**xgb_params)
    xgb_fold.fit(X_tr, y_tr)
    xgb_oof[val_idx] = xgb_fold.predict_proba(X_val)[:, 1]
    xgb_test_preds += xgb_fold.predict_proba(X_test)[:, 1] / n_splits
    
    # LightGBM fold
    lgb_fold = lgb.LGBMClassifier(**lgb_params)
    lgb_fold.fit(X_tr, y_tr)
    lgb_oof[val_idx] = lgb_fold.predict_proba(X_val)[:, 1]
    lgb_test_preds += lgb_fold.predict_proba(X_test)[:, 1] / n_splits
    
    # CatBoost fold
    cat_fold = CatBoostClassifier(**cat_params)
    cat_fold.fit(X_tr, y_tr)
    cat_oof[val_idx] = cat_fold.predict_proba(X_val)[:, 1]
    cat_test_preds += cat_fold.predict_proba(X_test)[:, 1] / n_splits


Building stacking ensemble...
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5


In [19]:
# Cell 8: Meta-learner with 4 models
meta_features_train = np.column_stack([xgb_oof, lgb_oof, cat_oof])
meta_features_test = np.column_stack([xgb_test_preds, lgb_test_preds, cat_test_preds])

meta_learner = LogisticRegression(random_state=42)
meta_learner.fit(meta_features_train, y_train)

stack_proba = meta_learner.predict_proba(meta_features_test)[:, 1]
stack_pred = (stack_proba > 0.5).astype(int)

stack_acc = accuracy_score(y_test, stack_pred)
stack_f1 = f1_score(y_test, stack_pred)
stack_auc = roc_auc_score(y_test, stack_proba)

print(f"\nStacking - Acc: {stack_acc:.4f}, F1: {stack_f1:.4f}, AUC: {stack_auc:.4f}")



Stacking - Acc: 0.7601, F1: 0.6098, AUC: 0.8046


In [20]:
# Cell 9: Results comparison
results = pd.DataFrame({
    'Model': ['XGBoost', 'LightGBM', 'CatBoost', 'Stacking'],
    'Accuracy': [xgb_acc, lgb_acc, cat_acc, stack_acc],
    'F1': [xgb_f1, lgb_f1, cat_f1, stack_f1],
    'ROC_AUC': [xgb_auc, lgb_auc, cat_auc, stack_auc]
})

print("\nFinal Results:")
print(results.to_string(index=False))


Final Results:
   Model  Accuracy       F1  ROC_AUC
 XGBoost  0.760167 0.609201 0.804806
LightGBM  0.760200 0.609912 0.804499
CatBoost  0.760133 0.609125 0.804383
Stacking  0.760133 0.609846 0.804553


In [21]:
# Cell 10: Meta-learner weights
print(f"\nMeta-learner coefficients:")
print(f"XGBoost: {meta_learner.coef_[0][0]:.4f}")
print(f"LightGBM: {meta_learner.coef_[0][1]:.4f}")
print(f"CatBoost: {meta_learner.coef_[0][2]:.4f}")


Meta-learner coefficients:
XGBoost: -2.9241
LightGBM: 5.2616
CatBoost: 3.5485


In [22]:
# Cell 9: Save
joblib.dump(xgb_model, '../models/xgboost_optimized.pkl')
joblib.dump(lgb_model, '../models/lightgbm_optimized.pkl')
joblib.dump(cat_model, '../models/catboost_optimized.pkl')
joblib.dump(meta_learner, '../models/meta_learner.pkl')
joblib.dump(results, '../models/final_results.pkl')

['../models/final_results.pkl']